<img src="img/swdb_logo.jpg" width="900">

<h1 align="center">Connectomics Exercise 1: Connection specificity</h1>
<h3 align="center">Summer Workshop on the Dynamic Brain 2026</h3>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2> Preliminaries and imports </h2>
<p>The cells below set up the Common Connectivity dataset reader, load the proofread cells, synapses, and cell types, and import the helper functions (<code>filter_synapse_table</code>, <code>make_adjacency</code>) used throughout this exercise. These are straight out of Module 2, refer back to that for more information.

</div>

In [ ]:
import sys
from os.path import join as pjoin
from typing import Optional, Union

mat_version = 1196

# Identifiers within the Common Connectivity dataset
project_id = "v1dd"
synapse_dataset_id = f"v1dd_{mat_version}_em"
synapse_feature_matrix_id = f"v1dd_{mat_version}_synapse_features"
axon_dataset_id = f"v1dd_{mat_version}_proofread_axons"
dendrite_dataset_id = f"v1dd_{mat_version}_proofread_dendrites"

sys.path.append(pjoin("..", "utils"))

from paths import resolve_data_root, resolve_dataset_dir

data_root = resolve_data_root(f"v1dd_{mat_version}_ccm")
ccm_dir = resolve_dataset_dir(f"v1dd_{mat_version}_ccm", root=data_root)

print(f"ccm_dir  {ccm_dir}")

In [ ]:
# Import packages
from connects_common_connectivity.io import DatasetReader, read_synapse_table

import pandas as pd
import polars as pl
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from scipy.sparse import csr_array
from typing import Union, Optional

# note: we are importing some of the functions that you saw in Module 2
from utils import adjacencyplot, cell_type_palette, filter_synapse_table, make_adjacency

In [ ]:
# Initialize the Common Connectivity dataset reader
reader = DatasetReader(ccm_dir)
reader.display_dataset_names() # see available cohorts

In [ ]:
# Loads cells with axon and dendrite proofreading
proofread_axons = reader.read_dataset(axon_dataset_id)
proofread_dendrites = reader.read_dataset(dendrite_dataset_id)

dendrite_proof_root_ids = proofread_dendrites[
    "dataitem_id"
].cast(pl.UInt64).to_numpy()
axon_proof_root_ids = proofread_axons["dataitem_id"].cast(pl.UInt64).to_numpy()

print(
    f"There are {len(dendrite_proof_root_ids)} cells with acceptable dendrites, and {len(axon_proof_root_ids)} cells with axon proofreading"
)

# get the ids of all proofread cells with both axon and dendrite proofreading
proof_root_ids = axon_proof_root_ids[
    np.isin(axon_proof_root_ids, dendrite_proof_root_ids)
]


In [ ]:
synapse_data = read_synapse_table(
    project_id,
    dataset_id=synapse_dataset_id,
    features=True,
    feature_matrix_id=synapse_feature_matrix_id,
    output_root=ccm_dir,
)


syn_df = (
    synapse_data.with_columns(
        pl.col("id").cast(pl.UInt64),
        pl.col("presynaptic_cell").cast(pl.UInt64),
        pl.col("postsynaptic_cell").cast(pl.UInt64),
    )
    .rename(
        {
            "presynaptic_cell": "pre_pt_root_id",
            "postsynaptic_cell": "post_pt_root_id",
        }
    )
    .select(
        [
            "id",
            "pre_pt_position_x",
            "pre_pt_position_y",
            "pre_pt_position_z",
            "post_pt_position_x",
            "post_pt_position_y",
            "post_pt_position_z",
            "ctr_pt_position_x",
            "ctr_pt_position_y",
            "ctr_pt_position_z",
            "size",
            "pre_pt_root_id",
            "post_pt_root_id",
        ]
    )
    .to_pandas()
)

print(syn_df.shape)
syn_df.set_index("id", inplace=True)
syn_df.head(3)

In [ ]:
cell_df = (
    proofread_dendrites
    .select(
        pl.col("dataitem_id").cast(pl.UInt64).alias("pt_root_id"),
        pl.col("soma_voxel_x").alias("pt_position_x"),
        pl.col("soma_voxel_y").alias("pt_position_y"),
        pl.col("soma_voxel_z").alias("pt_position_z"),
        pl.col("soma_transformed_x").alias("pt_position_trform_x"),
        pl.col("soma_transformed_y").alias("pt_position_trform_y"),
        pl.col("soma_transformed_z").alias("pt_position_trform_z"),
        pl.col("soma_volume").alias("volume"),
        pl.col("v1dd_cell_types_level_1").alias("cell_type_coarse"),
        pl.col("v1dd_cell_types_level_2").alias("cell_type"),
    )
    .to_pandas()
)
# Add a column that is soma depth in consistent units
cell_df['depth_um'] = cell_df['pt_position_trform_y'] 

cell_df.head()

In [ ]:
# Filter the proofread ids for those with a known cell type
proof_root_ids = np.intersect1d(proof_root_ids, cell_df["pt_root_id"].values)

# Filter the cell type table for those with a proofread root id
proof_cell_df = cell_df.set_index("pt_root_id").loc[proof_root_ids]
proof_cell_df = proof_cell_df.query("cell_type.notna()")

# NOTE: the adjacency matrix will be sorted according to this dataframe's index, so
# we'll sort it by soma depth
proof_cell_df = proof_cell_df.sort_values("depth_um")

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2> Connection specificity onto compartment types </h2>
<p>Up until now we have considered all synapses between any two cells, but different synapses target different compartments on the postsynaptic cell.</p>
    
<img src="../../figures/sss-diagram.png" width="600">

<p>For example, it is common understanding that:
<ol>
<li>Synapses between excitatory and excitatory cells are onto spines.
<li>Synapses between inhibitory and excitatory cells are onto dendritic shafts.
<li>Synapses onto inhibitory cells are onto dendritic shafts.
<li>Basket cells target soma synapses.
</ol>
<p>Let's test this textbook understanding in the real data.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 1: recreate the adjacency matrix plot from Module 2, but selected per target compartment</b>

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>To begin, we've loaded up the target structure information as a Pandas Series for you, indexed by synapse ID:

</div>

In [ ]:
target_structure = (
    synapse_data.select(pl.col("id").cast(pl.UInt64), "synaptictargetlabel")
    .to_pandas()
    .set_index("id")["synaptictargetlabel"]
)
target_structure

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Note that not all synapses will have predictions associated with them!
<p>In the cell below, use Pandas to include the target structure as a new column in your synapse data frame.

</div>

In [ ]:
# Attach the target-structure labels to syn_df as a new column, then count them.
# `target_structure` is a Series indexed by synapse id, and syn_df is indexed the same way.

syn_df['target_structure'] = ...

syn_df['target_structure'].value_counts()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Now, we are going to remake the <code>adjacencyplot</code> from Module 2, but showing only the connections for one type of target structure (spine, shaft, or soma) at a time.
<p>First, filter the synapse table to just connections among proofread cells:

</div>

In [ ]:
# Restrict the synapse table to connections where BOTH partners are proofread cells.
# `filter_synapse_table` takes pre_root_ids and post_root_ids.

proof_proof_syn_table = ...


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Then, filter down to just the synapses onto spine, and pass into <code>make_adjacency</code> to get your matrix to plot. The plotting code is repeated here for your convenience.

</div>

In [ ]:
category = 'spine' # This is synapses onto spines

# Select the synapses onto this compartment, then build the adjacency matrix from them.
# Use aggfunc="sum" so the matrix carries synapse size, as in Module 2.

category_syns_df = ...
category_adjacency = ...

In [ ]:
## Plot just synapses onto spine

fig, ax = plt.subplots(figsize=(8, 8), dpi=200)

# render the adjacency plot
adjacencyplot(
    category_adjacency,  
    nodes=proof_cell_df,  # data to organize the x and y axis
    groupby=["cell_type_coarse", "cell_type"],  # categorical variables to organize by
    sortby="pt_position_y",  # sort within groups by variable
    node_palette=cell_type_palette,
    title="Proofread connectivity",
    edge_palette="Greys",
    hue_norm=(0, 2000),  # normalize the color scale for the edges
    ax=ax,
    label_fontsize="xx-small",
    title_fontsize="medium",
    arc_labels=False
)


<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 2: Reproduce the plot above for <code>shaft</code> and <code>soma</code> instead of <code>spine</code>.</b>
<p>For loops or making a function may be helpful here!

</div>

In [ ]:
# Loop over the three compartments and draw one adjacency plot each.
# The body is the same as the two cells above -- select, make_adjacency, adjacencyplot.

for category in ['spine', 'shaft', 'soma']:
    ...

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Discussion</b>
<p>For each of the common cell-type connectivity assumptions above, which hold up in the data?
<p>Find one thing that does not match your expectation and write it on the whiteboard.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 3 (bonus): how does the distribution of spine input vary for E vs I cells?</b>
<p>Canonically, excitatory neurons are thought to have many more synaptic inputs onto spines than inhibitory cells. Does this hold up in our connectome data? And, how much variability is there within E or I? This may be related to variability in how much excitatory input a given excitatory neuron receives, for instance.

<p>For each proofread cell, calculate the proportion of its input synapses that are onto spine, shaft, and soma, and add these as new columns to <code>proofread_cell_df</code> below. There are many ways to accomplish this, but Pandas <code>df.groupby()</code> will probably be your friend!

<p>A suggested way to compare these distributions is a single plot showing two histograms (E vs. I cells in different colors) with proportion of spine input for each cell on the x-axis. 
<p>Hint: once you have the dataframe described above, this can be a one-liner with the right seaborn function!

</div>

In [ ]:
proofread_cell_df = cell_df.reset_index().set_index("pt_root_id").loc[proof_root_ids].copy()

In [ ]:
# For each proofread cell, what proportion of its INPUT synapses land on each
# compartment? Group the synapses by their postsynaptic cell and use
# value_counts(normalize=True) on target_structure.

cell_p_per_target = ...
cell_p_per_target

In [ ]:
# That gives a Series with a two-level index. Pivot the compartment level out into
# columns so there is one row per cell and one column per compartment.

cell_stats = ...
cell_stats

In [ ]:
# Join those proportions onto proofread_cell_df so cell type and compartment
# proportions sit in one table.

proofread_cell_df = ...
proofread_cell_df

In [ ]:
# One plot, two histograms: proportion of spine input on the x-axis, E and I in
# different colours. `cell_type_palette` already maps the coarse types to colours.

...

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 4 (bonus): find highly spiny or non-spiny excitatory cells</b>
<p>Out of the excitatory cells in our proofread set, which receive the most of their synapses onto spines? Which receive the least?

</div>

In [ ]:
# Among excitatory cells only, sort by the spine proportion and take the ends.

sorted_e_roots = ...
least_spiny = ...
most_spiny = ...

print("Least spiny:", least_spiny)
print("Most spiny:", most_spiny)

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 5: check our work — find these cells in neuroglancer and compare them</b>
<p>Make sure what we've done makes sense! In what ways do these cells resemble each other morphologically? In what ways are they different?

</div>